# Phase 23 — the calibration run

**Read `HANDOFF.md` §9 before changing anything here.** This notebook changes
exactly TWO things against Phase 16b's known-good configuration, and both are
there for a measured reason:

| change | why |
|---|---|
| `--init-checkpoint` | continue from PTify's own weights instead of restarting from ByteDance's. Without it every run discards what the last one learned. |
| `--loss-weights velocity=0.1` | velocity was **92.5% of 16b's total loss and moved +0.1%** -- nearly an additive constant that sets the gradient scale while contributing almost no signal. |

Plus `--batch-size 4 --accum-steps 2`, which keeps the effective batch at 8
while halving the accumulation steps. 16b peaked at **4.99 GB of a T4's
14.56 GB**, so the old 2x4 was an AMP-era OOM workaround that fp32 never
needed.

### What this run is actually testing

Phase 22 measured PTify's frame head directly and found it **miscalibrated,
not degraded**: its AUC essentially matches ByteDance's (0.9785 vs 0.9885)
while its activation *level* collapsed (median on sounding frames **0.347 vs
0.974**). The level moved **63x more than the ranking**. So HANDOFF's previous
advice -- 'weight the frame loss up' -- would have trained harder on a quantity
that had already improved 25.9% on validation.

**This is a ~10h de-risking run, not the long haul.** It exists to confirm the
new instrumentation reads correctly before committing weeks of quota. The
success criterion is stated in section 4, in advance.


In [ ]:
COMMIT = "phase-22-precision"   # a branch, tag, or full SHA
REPO = "https://github.com/ImSe4n/PTify.git"

# NOTE ON `!...`: the shell magics below build their command with an
# EXPLICIT f-string rather than relying on IPython's `!cmd {VAR}`
# interpolation. That interpolation did not fire on Kaggle -- the clone ran
# with a literal `{REPO}` and failed with "repository '{REPO}' does not
# exist". Building the string in Python first cannot fail that way, and the
# training cell uses the same form so it cannot fail that way ten hours in.

# --no-deps is load-bearing: Kaggle's preinstalled torch is much newer than
# this project's local pin (measured: torch 2.10 / numpy 2.0 there vs
# 2.2/1.26 here). That is FINE -- a checkpoint is a plain state_dict and
# crosses versions -- but letting a resolver loose would reinstall torch and
# waste most of the session.
!pip install -q --no-deps git+{REPO}@{COMMIT}
!pip install -q --no-deps piano_transcription_inference torchlibrosa \
    mido pretty_midi librosa soundfile resampy audioread soxr lazy_loader msgpack

import importlib
missing = []
for m in ["mido", "pretty_midi", "librosa", "soundfile", "resampy", "soxr",
          "torchlibrosa", "piano_transcription_inference"]:
    try:
        importlib.import_module(m)
    except Exception as e:
        missing.append("%s: %s %s" % (m, type(e).__name__, e))
print("MISSING:", missing or "none -- all imports OK")

import torch, numpy
print("torch", torch.__version__, "| numpy", numpy.__version__)
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


## 1. Clone the repo and find the MAESTRO mount


In [ ]:
# Clone as well as pip-install: pip installs the PACKAGES but not the repo's
# data files, and the segment index in benchmarks/ is a data file.
#
# NOT `-q`, and NOT `|| true`. Both hide the failure: `-q` suppresses git's
# error message and `|| true` reports success anyway, so the next cell fails on
# a missing file and points at the wrong thing. This cell must say WHY.

# --- is there internet at all? ------------------------------------------
# Kaggle disables internet by default on a new notebook, and every symptom of
# that looks like a broken URL: the clone fails, pip fails, and the 172MB
# checkpoint download in section 2 fails. It is a SETTINGS toggle, not a code
# problem -- Notebook -> Settings -> Internet -> On (needs a verified phone
# number on the account).
import socket

def _online(host="github.com", port=443, timeout=8):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError as exc:
        print("cannot reach %s:%s -- %s" % (host, port, exc))
        return False

ONLINE = _online()
print("internet:", "ON" if ONLINE else "OFF")
if not ONLINE:
    print()
    print("  Turn it on: right-hand panel -> Settings -> Internet -> On.")
    print("  (Kaggle requires a phone-verified account for this.)")
    print("  Nothing below can work until then -- the clone, pip, and the")
    print("  172MB checkpoint download in section 2 all need it.")

# --- clone, showing the real error --------------------------------------
!git clone --depth 1 --branch {COMMIT} {REPO} /kaggle/working/PTify

import pathlib
repo = pathlib.Path("/kaggle/working/PTify")

if not repo.is_dir():
    raise SystemExit(
        "clone failed -- read git's message directly above this line.\n"
        "  'Could not resolve host'      -> internet is OFF (see above)\n"
        "  'Remote branch ... not found' -> COMMIT=%r is not on the remote;\n"
        "                                   push the branch first\n"
        "  'repository ... not found'    -> REPO is wrong or private"
        % COMMIT
    )

index_files = sorted(repo.glob("benchmarks/*.json"))
assert index_files, "cloned, but benchmarks/*.json is missing"
for f in index_files:
    print("{:>12,}  {}".format(f.stat().st_size, f.name))

# The flags this run needs only exist on phase-22-precision and later. Fail
# here rather than after the 172MB download and a model load.
train_py = (repo / "training" / "train.py").read_text(encoding="utf-8")
for flag in ("--init-checkpoint", "--loss-weights"):
    assert flag in train_py, (
        "%s is missing from the cloned training/train.py -- COMMIT=%s is too "
        "old. Push the branch, or point COMMIT at one that has it."
        % (flag, COMMIT))
print("\nOK: cloned code supports --init-checkpoint and --loss-weights")

import glob, os
print("\n--- attached datasets ---")
for path in sorted(glob.glob("/kaggle/input/*/")):
    print(path)
    for sub in sorted(glob.glob(path + "*"))[:6]:
        print("   ", os.path.basename(sub))


In [ ]:
# Verified working with the `alonhaviv/the-maestro-dataset-v3-0-0` public
# dataset. Kaggle nests an attached dataset under the uploader's name, so this
# path is specific to that upload. What matters is that the directory CONTAINS
# the year folders (2004/ ... 2018/).
AUDIO_ROOT = "/kaggle/input/datasets/alonhaviv/the-maestro-dataset-v3-0-0/maestro-v3.0.0"

INDEX = "benchmarks/maestro_segments.json"

import json, pathlib
index = json.load(open("/kaggle/working/PTify/" + INDEX))
print("summary:", json.dumps(index["summary"], indent=2))

sample = index["tracks"][0]["audio_filename"]
probe = pathlib.Path(AUDIO_ROOT) / sample
print("\nprobe:", probe)
print("EXISTS:", probe.exists(), "<- must be True before going further")
print("MIDI :", (pathlib.Path(AUDIO_ROOT) / index["tracks"][0]["midi_filename"]).exists())


## 2. Fetch PTify's weights -- the starting point

**This is the new step, and it is what makes the run cumulative.**
`training/train.py` calls `load_pretrained()`, which always loads *ByteDance's*
checkpoint. Without `--init-checkpoint` this run would restart from the
baseline and throw away 16b's 6,555 steps.

The digest is checked here rather than trusted. The inference library validates
a checkpoint by **size alone** (>160MB), so any other ~172MB `.pth` would load
without complaint -- and Phase 18 caught the release carrying the 260MB
*training* checkpoint where the 172MB deployable was expected.


In [ ]:
import hashlib, urllib.request, pathlib

PTIFY_URL = ("https://github.com/ImSe4n/PTify/releases/download/"
             "model-v1/ptify-16b-step6555.pth")
PTIFY_SHA256 = "17286ad93c5806e02a59caf0333769d9bea9f4f3e53abd7360be8cabe9d4accd"
INIT_CKPT = "/kaggle/working/ptify-16b-step6555.pth"

if not pathlib.Path(INIT_CKPT).exists():
    # Needs internet ON -- see section 1. A URLError here is that toggle,
    # not a bad URL: the asset was verified live at 172,037,521 bytes.
    print("downloading 172MB ...")
    urllib.request.urlretrieve(PTIFY_URL, INIT_CKPT)

h = hashlib.sha256()
with open(INIT_CKPT, "rb") as fh:
    for chunk in iter(lambda: fh.read(1 << 20), b""):
        h.update(chunk)

size = pathlib.Path(INIT_CKPT).stat().st_size
print("size  : {:,} (expected 172,037,521)".format(size))
print("sha256:", h.hexdigest())
assert h.hexdigest() == PTIFY_SHA256, "WRONG WEIGHTS -- do not train on these"
print("\nOK: this is the 16b checkpoint, verified by digest.")


## 3. Train

~10 hours at the measured 0.28 steps/s. `--resume auto` means re-running this
cell after a session dies continues from the last checkpoint -- Kaggle caps
sessions at ~12h, so expect to do that at least once.

**Keep the `step_N.pt` files this time.** 16b's 260MB resumable checkpoint was
lost and only the 172MB deployable survived, which is the entire reason
`--init-checkpoint` had to be written.


In [ ]:
!cd /kaggle/working/PTify && python -m training.train \
    --index benchmarks/maestro_segments.json \
    --audio-root {AUDIO_ROOT} \
    --out /kaggle/working/checkpoints \
    --init-checkpoint {INIT_CKPT} \
    --loss-weights velocity=0.1 \
    --augment \
    --augment-seed 0 \
    --device cuda \
    --no-amp \
    --steps 10000 \
    --batch-size 4 \
    --accum-steps 2 \
    --workers 2 \
    --log-every 50 \
    --validate-every 500 \
    --val-batches 20 \
    --save-every-seconds 1800 \
    --keep-checkpoints 3 \
    --resume auto


## 4. Read the curves -- and what counts as success

**Read the PER-HEAD numbers, never `total`.** That is 16b's headline lesson:
velocity dominated the sum, so the augmented `total` moved -1.4% while the
three heads that decide note F1 moved -14.2%. Watching `total` made a working
run look stalled for hours. This run down-weights velocity so `total` is more
readable -- but the per-head values are still the meaningful ones, and they are
logged **unweighted** on purpose so they stay comparable with 16b's log.

### The success criterion, decided in advance

| signal | 16b did | this run should |
|---|---|---|
| `val_onset` | **+1.1% (got worse)** | stop rising -- the clearest read on whether down-weighting velocity helped |
| `val_frame` | -25.9% | keep falling |
| `val_offset` | -7.0% | keep falling |

**If `val_onset` is still rising at step 10,000, do not commit weeks of
quota** -- the velocity weight is not the lever, and something else is going on.

Also worth noting: establish the noise floor before reading any trend. The
20-batch validation carries about +/-0.003, and per-step training loss has a
spread an order of magnitude larger than real validation movement.


In [ ]:
import json

rows = [json.loads(l) for l in
        open("/kaggle/working/checkpoints/train_log.jsonl") if l.strip()]
train = [r for r in rows if "onset" in r]
val = [r for r in rows if "val_onset" in r]
train.sort(key=lambda r: r["step"])
val.sort(key=lambda r: r["step"])

print("train rows", len(train), " validation rows", len(val))
if train:
    print("steps/s:", train[-1].get("steps_per_s"),
          "| peak GPU GB:", max(r.get("gpu_mem_gb", 0) for r in train))

print()
header = ("step", "val_onset", "val_frame", "val_offset", "aug_frame")
print("%6s %10s %10s %11s %10s" % header)
for r in val:
    print("%6d %10.5f %10.5f %11.5f %10.5f" % (
        r["step"], r["val_onset"], r["val_frame"], r["val_offset"],
        r.get("val_aug_frame", float("nan"))))

if len(val) >= 2:
    a, b = val[0], val[-1]
    print("\n--- movement, first validation to last ---")
    for k in ("onset", "offset", "frame", "velocity"):
        x, y = a["val_" + k], b["val_" + k]
        print("  %-9s %.5f -> %.5f  %+6.1f%%" % (k, x, y, 100 * (y - x) / x))

    delta = 100 * (b["val_onset"] - a["val_onset"]) / a["val_onset"]
    print()
    if delta < 0:
        print("val_onset IMPROVED (%+.1f%%) -- the velocity weight helped." % delta)
    else:
        print("val_onset STILL RISING (%+.1f%%)." % delta)
        print("Do NOT commit weeks of quota. See section 4.")


## 5. Verify the artifact BEFORE trusting any score

`PianoTranscription.__init__` re-downloads any checkpoint under 160MB and
loads with `strict=False`, so a bad save is not an error -- it silently scores
**ByteDance's** weights under your filename, and reads exactly like 'training
didn't help'.


In [ ]:
import sys, os, hashlib
sys.path.insert(0, "/kaggle/working/PTify")

from training.model import assert_deployable

ckpt = "/kaggle/working/checkpoints/ptify-note-pedal.pth"
assert_deployable(ckpt)
print("OK:", ckpt, "{:,} bytes".format(os.path.getsize(ckpt)))


def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


# It must DIFFER from what we started with -- otherwise the run trained
# nothing and the artifact is just a copy of 16b under a new name.
print("start :", sha(INIT_CKPT)[:16])
print("result:", sha(ckpt)[:16])
assert sha(ckpt) != sha(INIT_CKPT), "identical to the input -- nothing trained"
print("\nthe weights genuinely moved.")


## 6. Download BOTH files

- `ptify-note-pedal.pth` (172MB) -- the deployable checkpoint, for scoring.
- `step_*.pt` (~290MB) -- the **resumable** checkpoint. Keep it this time.

### Then, on the laptop -- in this order

```bash
# 1. RECALIBRATE FIRST. A retrained head invalidates both 0.01 and 0.6, and
#    scoring through stale thresholds misattributes a decode artifact to the
#    weights -- exactly the mistake Phase 19 undid.
set PTIFY_CHECKPOINT=C:\path\to\ptify-note-pedal.pth
python -m tools.calibrate_thresholds --audio-dir recordings/maps_paired ^
    --engine ptify --limit 6

# 2. THEN score, and diff on onset_p as well as onset_f1 -- precision is the
#    metric that says whether garbage notes actually fell.
set PYTHONUNBUFFERED=1
python -m evaluation --audio-dir recordings/maps_paired --engine ptify ^
    --preset clean --json benchmarks/real/maps-paired-ptify23-clean.json
```

Compare against `benchmarks/real/maps-paired-ptify17-clean.json` (0.8395), but
note it was scored at `onset_threshold=0.3` -- Phase 22 measured 0.6 for ptify,
so re-score the baseline at the new value before reading the delta.


In [ ]:
from IPython.display import FileLink
import glob, os

for path in ["/kaggle/working/checkpoints/ptify-note-pedal.pth",
             "/kaggle/working/checkpoints/train_log.jsonl"]:
    if os.path.exists(path):
        display(FileLink(path))

# The RESUMABLE checkpoints. 16b's was lost and only the deployable survived,
# which is the entire reason --init-checkpoint had to be written. Keep these.
for p in sorted(glob.glob("/kaggle/working/checkpoints/step_*.pt")):
    print("{:,}".format(os.path.getsize(p)), p)
    display(FileLink(p))
